In [4]:
!python ../../mmdetection3d/demo/pcd_demo.py ../../mmdetection3d/demo/data/sunrgbd/000017.bin ../../mmdetection3d/configs/votenet/votenet_8xb16_sunrgbd-3d.py "/home/jvermandere/projects/DRM/checkpoints/votenet_16x8_sunrgbd-3d-10class_20210820_162823-bf11f014.pth"

/home/jvermandere/.conda/envs/openmmlab/lib/python3.8/site-packages/mmcv/cnn/bricks/conv_module.py:208: UserWarning: Unnecessary conv bias before batch/instance norm
  warnings.warn(
Loads checkpoint by local backend from path: /home/jvermandere/projects/DRM/checkpoints/votenet_16x8_sunrgbd-3d-10class_20210820_162823-bf11f014.pth
02/25 16:05:47 - mmengine - WARNING - Failed to search registry with scope "mmdet3d" in the "function" registry tree. As a workaround, the current "function" registry in "mmengine" is used to build instance. This may cause unexpected failure when running the built modules. Please check whether "mmdet3d" is a correct scope, or whether the registry is initialized.
/home/jvermandere/.conda/envs/openmmlab/lib/python3.8/site-packages/mmengine/visualization/visualizer.py:196: UserWarning: Failed to add <class 'mmengine.visualization.vis_backend.LocalVisBackend'>, please provide the `save_dir` argument.
  warnings.warn(f'Failed to add {vis_backend.__class__}, '
/home

In [ ]:
# the path of the pointcloud
dataPath = "../../mmdetection3d/demo/data/sunrgbd/000017.bin"

demoFile = "../../mmdetection3d/demo/pcd_demo.py"
configFile = "../../mmdetection3d/configs/votenet/votenet_8xb16_sunrgbd-3d.py"
weightsFile = "/home/jvermandere/projects/DRM/checkpoints/votenet_16x8_sunrgbd-3d-10class_20210820_162823-bf11f014.pth"

!python {demoFile} {dataPath} {configFile} {weightsFile}

/home/jvermandere/.conda/envs/openmmlab/lib/python3.8/site-packages/mmcv/cnn/bricks/conv_module.py:208: UserWarning: Unnecessary conv bias before batch/instance norm
  warnings.warn(
Loads checkpoint by local backend from path: /home/jvermandere/projects/DRM/checkpoints/votenet_16x8_sunrgbd-3d-10class_20210820_162823-bf11f014.pth
02/25 16:12:10 - mmengine - WARNING - Failed to search registry with scope "mmdet3d" in the "function" registry tree. As a workaround, the current "function" registry in "mmengine" is used to build instance. This may cause unexpected failure when running the built modules. Please check whether "mmdet3d" is a correct scope, or whether the registry is initialized.
/home/jvermandere/.conda/envs/openmmlab/lib/python3.8/site-packages/mmengine/visualization/visualizer.py:196: UserWarning: Failed to add <class 'mmengine.visualization.vis_backend.LocalVisBackend'>, please provide the `save_dir` argument.
  warnings.warn(f'Failed to add {vis_backend.__class__}, '
/home

In [1]:
import mmdet3d

In [7]:
from mmdet3d.apis import init_model, inference_detector

config_file = 'pointpillars_hv_secfpn_8xb6-160e_kitti-3d-car.py'
checkpoint_file = 'hv_pointpillars_secfpn_6x8_160e_kitti-3d-car_20220331_134606-d42d15ed.pth'
model = init_model(config_file, checkpoint_file)
inference_detector(model, 'demo/data/kitti/000008.bin')

Loads checkpoint by local backend from path: hv_pointpillars_secfpn_6x8_160e_kitti-3d-car_20220331_134606-d42d15ed.pth


(<Det3DDataSample(
 
     META INFORMATION
     box_mode_3d: <Box3DMode.LIDAR: 0>
     pcd_scale_factor: 1.0
     box_type_3d: <class 'mmdet3d.structures.bbox_3d.lidar_box3d.LiDARInstance3DBoxes'>
     flip: False
     pcd_rotation_angle: 0.0
     transformation_3d_flow: ['R', 'S', 'T']
     pcd_vertical_flip: False
     pcd_trans: array([0., 0., 0.])
     lidar_path: 'demo/data/kitti/000008.bin'
     pcd_horizontal_flip: False
     pcd_rotation: tensor([[1., 0., 0.],
                 [-0., 1., 0.],
                 [0., 0., 1.]])
     axis_align_matrix: array([[1., 0., 0., 0.],
                [0., 1., 0., 0.],
                [0., 0., 1., 0.],
                [0., 0., 0., 1.]])
 
     DATA FIELDS
     pred_instances_3d: <InstanceData(
         
             META INFORMATION
         
             DATA FIELDS
             labels_3d: tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0], device='cuda:0')
             scores_3d: tensor([0.9750, 0.9682, 0.9457, 0.8905, 0.8890, 0.7712, 0.7554, 0.7059, 0.

In [4]:
import torch
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())

2.1.2+cu121
12.1
True


In [5]:
import mmcv
print(mmcv.__version__)


2.1.0


In [6]:
from mmdet3d.apis import init_model, inference_detector
print("MMDet3D import OK ✅")

MMDet3D import OK ✅


In [1]:
# Copyright (c) OpenMMLab. All rights reserved.
import argparse
import tempfile

import torch
from mmengine import Config
from mmengine.runner import load_state_dict

from mmdet3d.registry import MODELS


def parse_args():
    parser = argparse.ArgumentParser(
        description='MMDet3D upgrade model version(before v0.6.0) of VoteNet')
    parser.add_argument('checkpoint', help='checkpoint file')
    parser.add_argument('--out', help='path of the output checkpoint file')
    args = parser.parse_args()
    return args


def parse_config(config_strings):
    """Parse config from strings.

    Args:
        config_strings (string): strings of model config.

    Returns:
        Config: model config
    """
    temp_file = tempfile.NamedTemporaryFile()
    config_path = f'{temp_file.name}.py'
    with open(config_path, 'w') as f:
        f.write(config_strings)

    config = Config.fromfile(config_path)

    # Update backbone config
    if 'pool_mod' in config.model.backbone:
        config.model.backbone.pop('pool_mod')

    if 'sa_cfg' not in config.model.backbone:
        config.model.backbone['sa_cfg'] = dict(
            type='PointSAModule',
            pool_mod='max',
            use_xyz=True,
            normalize_xyz=True)

    if 'type' not in config.model.bbox_head.vote_aggregation_cfg:
        config.model.bbox_head.vote_aggregation_cfg['type'] = 'PointSAModule'

    # Update bbox_head config
    if 'pred_layer_cfg' not in config.model.bbox_head:
        config.model.bbox_head['pred_layer_cfg'] = dict(
            in_channels=128, shared_conv_channels=(128, 128), bias=True)

    if 'feat_channels' in config.model.bbox_head:
        config.model.bbox_head.pop('feat_channels')

    if 'vote_moudule_cfg' in config.model.bbox_head:
        config.model.bbox_head['vote_module_cfg'] = config.model.bbox_head.pop(
            'vote_moudule_cfg')

    if config.model.bbox_head.vote_aggregation_cfg.use_xyz:
        config.model.bbox_head.vote_aggregation_cfg.mlp_channels[0] -= 3

    temp_file.close()

    return config



In [ ]:
checkpointPath = "/home/jvermandere/projects/DRM/checkpoints/votenet_8x8_scannet-3d-18class_20210823_234503-cf8134fa.pth"
checkpoint = torch.load(checkpointPath)
cfg = parse_config(checkpoint['meta']['config'])
# Build the model and load checkpoint
model = MODELS.build(
    cfg.model,
    train_cfg=cfg.get('train_pipeline'),
    test_cfg=cfg.get('test_pipeline'))
orig_ckpt = checkpoint['state_dict']
converted_ckpt = orig_ckpt.copy()

if cfg['dataset_type'] == 'ScanNetDataset':
    NUM_CLASSES = 18
elif cfg['dataset_type'] == 'SUNRGBDDataset':
    NUM_CLASSES = 10
else:
    raise NotImplementedError

RENAME_PREFIX = {
    'bbox_head.conv_pred.0': 'bbox_head.conv_pred.shared_convs.layer0',
    'bbox_head.conv_pred.1': 'bbox_head.conv_pred.shared_convs.layer1'
}

DEL_KEYS = [
    'bbox_head.conv_pred.0.bn.num_batches_tracked',
    'bbox_head.conv_pred.1.bn.num_batches_tracked'
]

EXTRACT_KEYS = {
    'bbox_head.conv_pred.conv_cls.weight':
    ('bbox_head.conv_pred.conv_out.weight', [(0, 2), (-NUM_CLASSES, -1)]),
    'bbox_head.conv_pred.conv_cls.bias':
    ('bbox_head.conv_pred.conv_out.bias', [(0, 2), (-NUM_CLASSES, -1)]),
    'bbox_head.conv_pred.conv_reg.weight':
    ('bbox_head.conv_pred.conv_out.weight', [(2, -NUM_CLASSES)]),
    'bbox_head.conv_pred.conv_reg.bias':
    ('bbox_head.conv_pred.conv_out.bias', [(2, -NUM_CLASSES)])
}

# Delete some useless keys
for key in DEL_KEYS:
    converted_ckpt.pop(key)

# Rename keys with specific prefix
RENAME_KEYS = dict()
for old_key in converted_ckpt.keys():
    for rename_prefix in RENAME_PREFIX.keys():
        if rename_prefix in old_key:
            new_key = old_key.replace(rename_prefix,
                                        RENAME_PREFIX[rename_prefix])
            RENAME_KEYS[new_key] = old_key
for new_key, old_key in RENAME_KEYS.items():
    converted_ckpt[new_key] = converted_ckpt.pop(old_key)

# Extract weights and rename the keys
for new_key, (old_key, indices) in EXTRACT_KEYS.items():
    cur_layers = orig_ckpt[old_key]
    converted_layers = []
    for (start, end) in indices:
        if end != -1:
            converted_layers.append(cur_layers[start:end])
        else:
            converted_layers.append(cur_layers[start:])
    converted_layers = torch.cat(converted_layers, 0)
    converted_ckpt[new_key] = converted_layers
    if old_key in converted_ckpt.keys():
        converted_ckpt.pop(old_key)

# Check the converted checkpoint by loading to the model
load_state_dict(model, converted_ckpt, strict=True)
checkpoint['state_dict'] = converted_ckpt
torch.save(checkpoint, "/home/jvermandere/projects/DRM/checkpoints/mmdet3d-votenet.pth")

TypeError: build_model_from_cfg() got an unexpected keyword argument 'train_cfg'

In [9]:
checkpointPath = "/home/jvermandere/projects/DRM/checkpoints/votenet_8x8_scannet-3d-18class_20210823_234503-cf8134fa.pth"
checkpoint = torch.load(checkpointPath)
cfg = parse_config(checkpoint['meta']['config'])

In [3]:
print(cfg.dump())

checkpoint_config = dict(interval=1, max_keep_ckpts=5)
class_names = (
    'cabinet',
    'bed',
    'chair',
    'sofa',
    'table',
    'door',
    'window',
    'bookshelf',
    'picture',
    'counter',
    'desk',
    'curtain',
    'refrigerator',
    'showercurtrain',
    'toilet',
    'sink',
    'bathtub',
    'garbagebin',
)
data = dict(
    samples_per_gpu=8,
    test=dict(
        ann_file='./data/scannet/scannet_infos_val.pkl',
        box_type_3d='Depth',
        classes=(
            'cabinet',
            'bed',
            'chair',
            'sofa',
            'table',
            'door',
            'window',
            'bookshelf',
            'picture',
            'counter',
            'desk',
            'curtain',
            'refrigerator',
            'showercurtrain',
            'toilet',
            'sink',
            'bathtub',
            'garbagebin',
        ),
        data_root='./data/scannet/',
        pipeline=[
            dict(
             

In [5]:
print(checkpoint)

{'meta': {'mmdet_version': '2.14.0', 'mmseg_version': '0.14.1', 'mmdet3d_version': '0.16.0', 'config': "dataset_type = 'ScanNetDataset'\ndata_root = './data/scannet/'\nclass_names = ('cabinet', 'bed', 'chair', 'sofa', 'table', 'door', 'window',\n               'bookshelf', 'picture', 'counter', 'desk', 'curtain',\n               'refrigerator', 'showercurtrain', 'toilet', 'sink', 'bathtub',\n               'garbagebin')\ntrain_pipeline = [\n    dict(\n        type='LoadPointsFromFile',\n        coord_type='DEPTH',\n        shift_height=True,\n        load_dim=6,\n        use_dim=[0, 1, 2]),\n    dict(\n        type='LoadAnnotations3D',\n        with_bbox_3d=True,\n        with_label_3d=True,\n        with_mask_3d=True,\n        with_seg_3d=True),\n    dict(type='GlobalAlignment', rotation_axis=2),\n    dict(\n        type='PointSegClassMapping',\n        valid_cat_ids=(3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14, 16, 24, 28, 33, 34,\n                       36, 39),\n        max_cat_id=40),\n  